# CSET485 Assignment 1 — Adult Census Income
**Udit Malik | S24CSEU0099**

Reproducible analysis notebook for model comparison, subgroup effects, omitted-variable sensitivity, confident mistakes, and fairness threshold trade-offs. Run all cells before writing the report; do not report unverified results.


In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
SEED=99
OUT=Path('../reports'); OUT.mkdir(parents=True, exist_ok=True)
columns=['age','workclass','fnlwgt','education','education_num','marital_status','occupation','relationship','race','sex','capital_gain','capital_loss','hours_per_week','native_country','income']
url='https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
df=pd.read_csv(url,header=None,names=columns,na_values='?',skipinitialspace=True)
df['income']=df['income'].astype(str).str.strip().str.replace('.','',regex=False)
df['income_gt50k']=(df['income']=='>50K').astype(int)
analysis_df=df.copy()
print('Shape:',df.shape,'\nSeed:',SEED,'\nTarget counts:'); display(df['income'].value_counts(dropna=False)); display(df.isna().sum().sort_values(ascending=False).head(10))


## 1. Preprocessing and train/test split
The split is 70/30 with `random_state=99` and stratification. Preprocessing is fitted on training data only. Numeric missing values use the median; categorical missing values use the most frequent value; numeric variables are standardized and categorical variables one-hot encoded. No rows are dropped just for being outliers.


In [ ]:
X=df.drop(columns=['income','income_gt50k']); y=df['income_gt50k']
num_cols=X.select_dtypes(include=np.number).columns.tolist(); cat_cols=X.select_dtypes(exclude=np.number).columns.tolist()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.30,random_state=SEED,stratify=y)
prep=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num_cols),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat_cols)])
Xtr=prep.fit_transform(X_train); Xte=prep.transform(X_test)
print(f'Train: {len(X_train)} ({len(X_train)/len(df):.1%}); test: {len(X_test)} ({len(X_test)/len(df):.1%}')
print('Numeric:',num_cols,'\nCategorical:',cat_cols)


## 2. Part 1 — OLS versus logistic regression
OLS predictions are thresholded at their test-set median; logistic predictions use probability 0.5. The confusion matrix compares the two classifiers, not either classifier against ground truth.


In [ ]:
dense=lambda a: a.toarray() if hasattr(a,'toarray') else np.asarray(a)
ols=sm.OLS(np.asarray(y_train),sm.add_constant(dense(Xtr),has_constant='add')).fit()
ols_pred=np.asarray(ols.predict(sm.add_constant(dense(Xte),has_constant='add'))); ols_median=float(np.median(ols_pred)); ols_class=(ols_pred>=ols_median).astype(int)
logit=LogisticRegression(max_iter=2000,solver='liblinear',random_state=SEED).fit(Xtr,y_train)
prob=logit.predict_proba(Xte)[:,1]; pred=(prob>=.5).astype(int)
mismatch=int((ols_class!=pred).sum()); mismatch_pct=100*mismatch/len(y_test)
print('OLS prediction median:',ols_median); print(f'Mismatch: {mismatch}/{len(y_test)} ({mismatch_pct:.2f}%)')
display(pd.DataFrame(confusion_matrix(ols_class,pred,labels=[0,1]),index=['OLS low','OLS high'],columns=['Logistic low','Logistic high']))
print(classification_report(y_test,pred,zero_division=0)); print('ROC AUC:',roc_auc_score(y_test,prob))
pd.DataFrame({'actual':np.asarray(y_test),'ols_prediction':ols_pred,'ols_class':ols_class,'logistic_probability':prob,'logistic_class':pred}).to_csv(OUT/'part1_predictions.csv',index=False)


## 3. Part 2.1 and 2.2 — Subgroups and omitted-variable sensitivity
Statsmodels logistic regressions compare the `education_num` coefficient in the full sample, Female subgroup, and Male subgroup. The omitted-variable test removes `education_num`; coefficient shifts indicate model sensitivity, not proof of causality. The redundant text `education` label is excluded while numeric `education_num` is retained.


In [ ]:
def fit_logit(frame, drop_cols=()):
    predictors=[c for c in frame.columns if c not in ({'income_gt50k','income','education'}|set(drop_cols))]
    xx=pd.get_dummies(frame[predictors],drop_first=True,dtype=float); xx=sm.add_constant(xx,has_constant='add')
    return sm.Logit(frame['income_gt50k'].astype(int),xx).fit(disp=False,maxiter=250)
fits={'Full':fit_logit(analysis_df),'Female':fit_logit(analysis_df[analysis_df.sex=='Female']),'Male':fit_logit(analysis_df[analysis_df.sex=='Male'])}
sub=[]
for group,fit in fits.items():
    ci=fit.conf_int().loc['education_num']; sub.append({'group':group,'education_num_coefficient':fit.params['education_num'],'ci_low':ci.iloc[0],'ci_high':ci.iloc[1],'p_value':fit.pvalues['education_num']})
subgroups=pd.DataFrame(sub); display(subgroups); subgroups.to_csv(OUT/'part2_1_subgroup_coefficients.csv',index=False)
full=fit_logit(analysis_df); reduced=fit_logit(analysis_df,drop_cols=['education_num'])
shared=[c for c in full.params.index if c in reduced.params.index and c!='const']; preferred=[c for c in ['age','fnlwgt','capital_gain','capital_loss','hours_per_week'] if c in shared]; chosen=(preferred+[c for c in shared if c not in preferred])[:8]
bias=pd.DataFrame({'predictor':chosen,'full_coefficient':[full.params[c] for c in chosen],'reduced_coefficient':[reduced.params[c] for c in chosen]})
bias['absolute_change']=(bias.reduced_coefficient-bias.full_coefficient).abs(); bias['percent_change']=np.where(bias.full_coefficient.abs()>1e-10,100*(bias.reduced_coefficient-bias.full_coefficient)/bias.full_coefficient,np.nan)
display(bias); bias.to_csv(OUT/'part2_2_omitted_variable.csv',index=False)


## 4. Part 2.3 — Confident mistakes
Confident false positives have probability at least 0.75 but actual class 0; confident false negatives have probability at most 0.25 but actual class 1. At most 50 are sampled with a fixed seed for inspection.


In [ ]:
test_view=X_test.reset_index(drop=True).copy(); test_view['actual']=np.asarray(y_test); test_view['probability']=prob; test_view['predicted']=pred
test_view['error_type']=np.select([(test_view.probability>=.75)&(test_view.actual==0),(test_view.probability<=.25)&(test_view.actual==1)],['confident_false_positive','confident_false_negative'],default='')
wrong=test_view[test_view.error_type!=''].copy(); adversarial=wrong.sample(n=min(50,len(wrong)),random_state=SEED) if len(wrong) else wrong.copy()
print(f'Confident errors: {len(wrong)}/{len(test_view)} ({100*len(wrong)/len(test_view):.2f}%)'); display(adversarial.head(20))
print('Age summary:'); display(pd.DataFrame({'test':test_view.age.describe(),'confident_errors':adversarial.age.describe()}))
print('Sex share, test:'); display(test_view.sex.value_counts(normalize=True).mul(100)); print('Sex share, confident errors:'); display(adversarial.sex.value_counts(normalize=True).mul(100))
adversarial.to_csv(OUT/'part2_3_adversarial_subset.csv',index=False)


## 5. Part 2.4 — Fairness threshold sweep
Ratios use Group A / Group B, where Group A is Female and Group B is Male. DP is selection-rate ratio; EO ratio is the minimum of the TPR and FPR ratios (the assignment's simplified definition); PP is positive predictive value ratio. Undefined zero-denominator ratios are recorded as NaN.


In [ ]:
def ratio(a,b): return np.nan if pd.isna(a) or pd.isna(b) or b==0 else a/b
sex=X_test.sex.reset_index(drop=True).to_numpy(); yy=np.asarray(y_test); rows=[]
for t in np.round(np.arange(.05,.951,.1),2):
    pp=(prob>=t).astype(int); g={}
    for name in ['Female','Male']:
        m=sex==name; yt,pt=yy[m],pp[m]; pos=pt.mean() if len(pt) else np.nan
        tpr=((pt==1)&(yt==1)).sum()/(yt==1).sum() if (yt==1).sum() else np.nan
        fpr=((pt==1)&(yt==0)).sum()/(yt==0).sum() if (yt==0).sum() else np.nan
        ppv=((pt==1)&(yt==1)).sum()/(pt==1).sum() if (pt==1).sum() else np.nan
        g[name]={'selection':pos,'TPR':tpr,'FPR':fpr,'PPV':ppv}
    a,b=g['Female'],g['Male']; r_tpr=ratio(a['TPR'],b['TPR']); r_fpr=ratio(a['FPR'],b['FPR'])
    rows.append({'threshold':t,'DP_ratio':ratio(a['selection'],b['selection']),'EO_ratio':min(r_tpr,r_fpr) if pd.notna(r_tpr) and pd.notna(r_fpr) else np.nan,'PP_ratio':ratio(a['PPV'],b['PPV']),'female_selection_rate':a['selection'],'male_selection_rate':b['selection'],'female_TPR':a['TPR'],'male_TPR':b['TPR'],'female_FPR':a['FPR'],'male_FPR':b['FPR'],'female_PPV':a['PPV'],'male_PPV':b['PPV'],'accuracy':accuracy_score(yy,pp)})
fair=pd.DataFrame(rows); display(fair.round(4)); fair.to_csv(OUT/'part2_4_fairness_thresholds.csv',index=False)
plt.figure(figsize=(8,6)); plt.scatter(fair.DP_ratio,fair.EO_ratio)
for _,r in fair.iterrows():
    if pd.notna(r.DP_ratio) and pd.notna(r.EO_ratio): plt.annotate(f'{r.threshold:.2f}',(r.DP_ratio,r.EO_ratio),xytext=(4,4),textcoords='offset points')
plt.xlim(0,1.5); plt.ylim(0,1.5); plt.xlabel('Demographic parity ratio (Female / Male)'); plt.ylabel('Equalized odds ratio (assignment definition)'); plt.title('Fairness trade-off across thresholds'); plt.grid(True,alpha=.3); plt.tight_layout(); plt.savefig(OUT/'fairness_tradeoff.png',dpi=160); plt.show()


In [ ]:
summary={'seed':SEED,'n_total':len(df),'n_train':len(X_train),'n_test':len(X_test),'ols_test_median':ols_median,'part1_mismatch_count':mismatch,'part1_mismatch_percent':mismatch_pct,'logistic_accuracy':accuracy_score(y_test,pred),'logistic_precision':precision_score(y_test,pred,zero_division=0),'logistic_recall':recall_score(y_test,pred,zero_division=0),'logistic_f1':f1_score(y_test,pred,zero_division=0),'logistic_roc_auc':roc_auc_score(y_test,prob),'confident_wrong_count':len(wrong),'confident_wrong_percent_of_test':100*len(wrong)/len(test_view)}
display(pd.Series(summary,name='value').to_frame()); pd.Series(summary,name='value').to_csv(OUT/'summary_metrics.csv')
print('Saved output tables and chart to',OUT.resolve())


## Report checklist
Use actual output values to write eight approximately 250–300 word answers and an approximately 500-word reflection. Discuss the OLS median and mismatch; subgroup coefficients and confidence intervals; omitted-variable shifts; a concrete method to investigate hidden confounding; observed confident errors; fairness trade-offs at multiple thresholds; a defended threshold using at least three metrics; and demographic parity versus equalized odds. Do not claim Simpson's paradox unless the estimates support it. Export and review the final PDF; do not leave placeholders.

**Dataset source:** [UCI Adult dataset](https://archive.ics.uci.edu/ml/machine-learning-databases/adult/).
